## Cropping and Extracting Audio-Script

In [4]:
import os
import re
import subprocess

# --- 1. SETUP ---
cha_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2334_T4.cha"
video_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2334_T4.mp4"
output_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2"

os.makedirs(output_dir, exist_ok=True)

# --- 2. REGEX PATTERNS ---
# This strictly looks for the timestamp bullets: 1917578_1923343
timestamp_pattern = re.compile(r"(\d+)_(\d+)")

clip_count = 0
print(f"🎧 Scanning {cha_file} and extracting 22050Hz audio...")

# --- 3. PARSING & EXTRACTION ---
with open(cha_file, "r", encoding="utf-8") as f:
    for line in f:
        # ONLY look at the Patient's lines
        if line.startswith("*PAR:"):
            match = timestamp_pattern.search(line)
            
            if match:
                start_ms = int(match.group(1))
                end_ms = int(match.group(2))
                
                # Convert to seconds
                start_sec = start_ms / 1000.0
                duration_sec = (end_ms - start_ms) / 1000.0
                
                # --- CLEAN THE TRANSCRIPT ---
                # Remove the '*PAR:' prefix and the timestamp bullet to isolate the text
                clean_text = line.replace("*PAR:", "").split("")[0].strip()
                
                # Skip completely empty lines just in case
                if not clean_text:
                    continue

                # --- 4. FILE PATHS ---
                base_filename = f"patient1_{clip_count:04d}"
                out_wav = os.path.join(output_dir, f"{base_filename}.wav")
                out_txt = os.path.join(output_dir, f"{base_filename}.txt")
                
                # --- 5. SAVE THE TRANSCRIPT ---
                with open(out_txt, "w", encoding="utf-8") as txt_file:
                    txt_file.write(clean_text)
                
                # --- 6. CROP & CONVERT AUDIO WITH FFMPEG ---
                cmd = [
                    "ffmpeg", "-y", 
                    "-ss", str(start_sec), 
                    "-i", video_file, 
                    "-t", str(duration_sec),
                    "-ar", "22050",          # Resample directly to 22050 Hz
                    "-ac", "1",              # Convert to Mono audio
                    "-c:a", "pcm_s16le",     # Uncompressed 16-bit WAV format
                    "-vn",                   # Strip the video track completely
                    out_wav
                ]
                
                # Run FFmpeg silently
                subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                
                clip_count += 1

print(f"\n✅ Done! Extracted {clip_count} audio files and transcripts to the '{output_dir}' folder.")

🎧 Scanning /rds/general/user/ak8224/home/emoji_project/data/Aphasia/2334_T4.cha and extracting 22050Hz audio...

✅ Done! Extracted 808 audio files and transcripts to the '/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2' folder.


## Pre-process Script

In [7]:
import os
import re
import glob

def clean_aphasia_transcript(text):
    """
    Strips out AphasiaBank CHAT formatting but keeps the raw spoken words.
    """
    # 1. Strip out the patient/other/interviewer label if it accidentally got left in
    text = text.replace("*PAR:", "")
    text = re.sub(r'&\*[A-Z]+:', '', text)
    
    # 2. Strip out timestamp bullets (e.g., 14328_26169)
    text = re.sub(r'\d+_\d+', '', text)
    
    # 3. Target the prefixes (&- and &+) but KEEP the attached filler words
    # Converts "&-um" to "um" and "&+i" to "i"
    text = re.sub(r'&[-+]', '', text)
    
    # 4. Remove all bracketed syntax (like [/], [//], or [* m:0s])
    text = re.sub(r'\[.*?\]', '', text)
    
    # 5. Remove angle brackets < and > used for grouping phrases
    text = re.sub(r'[<>]', '', text)
    
    # 6. Remove CHAT pause markers like (.), (..), (...) 
    text = re.sub(r'\(\.{1,3}\)', '', text)
    
    # 7. Clean up any weird double spaces left behind by the deleted symbols
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [9]:

# --- QUICK TEST PRINT ---
example = "&-um &-uh I'm [/] &+i I'm <going to> [//] &-uh in Dothan [/] &-um &*INV:Dothan Alabama . 14328_26169"
print(f"\n🔍 Test Original: {example}")
print(f"✨ Test Cleaned:  {clean_aphasia_transcript(example)}")


🔍 Test Original: &-um &-uh I'm [/] &+i I'm <going to> [//] &-uh in Dothan [/] &-um &*INV:Dothan Alabama . 14328_26169
✨ Test Cleaned:  um uh I'm i I'm going to uh in Dothan um Dothan Alabama .


In [10]:
# --- BATCH PROCESS DIRECTORY ---
dataset_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/2"
txt_files = glob.glob(f"{dataset_dir}/*.txt")

print(f"🧹 Found {len(txt_files)} transcripts to clean. Starting...")

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    cleaned_text = clean_aphasia_transcript(raw_text)
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

print("✅ All transcripts successfully sanitized!")

🧹 Found 808 transcripts to clean. Starting...
✅ All transcripts successfully sanitized!


## Pre-Process Audio

In [4]:
import os
import glob
from pydub import AudioSegment
from pydub.silence import split_on_silence

def preprocess_aphasia_audio(input_dir):
    """
    Normalizes volume to -23 dBFS and perfectly crops long silences down to 1 second.
    """
    output_dir = os.path.join(input_dir, "preprocessed")
    os.makedirs(output_dir, exist_ok=True)

    # Grab all your handpicked wav files
    wav_files = glob.glob(os.path.join(input_dir, "*.wav"))
    print(f"🎧 Found {len(wav_files)} files. Starting normalization and silence cropping...\n")

    for file_path in wav_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(output_dir, filename)

        # --- 1. LOAD AUDIO ---
        audio = AudioSegment.from_wav(file_path)
        original_duration = len(audio) / 1000.0

        # --- 2. VOLUME NORMALIZATION ---
        # We target -23 dBFS, which acts as a fantastic LUFS equivalent for mono voice tracks
        target_dBFS = -23.0
        gain_change = target_dBFS - audio.dBFS
        normalized_audio = audio.apply_gain(gain_change)

        # --- 3. DETECT & CROP SILENCE ---
        # min_silence_len=1000: We only trigger on silences longer than 1 second
        # silence_thresh=-40: Anything quieter than -40 dBFS is considered "dead air"
        # keep_silence=250: This is the magic trick. It leaves a 250ms "buffer" of room tone 
        # on the edges of the spoken words so we don't accidentally chop off natural breath sounds.
        chunks = split_on_silence(
            normalized_audio,
            min_silence_len=1000,
            silence_thresh=-40,
            keep_silence=250 
        )

        # --- 4. RE-STITCH WITH 1 SECOND GAP ---
        if chunks:
            final_audio = chunks[0]
            
            # Since we kept 250ms of breath on the tail of Chunk A and 250ms on the head of Chunk B,
            # we inject exactly 500ms of pure silence between them to equal a perfect 1.0 second pause.
            one_sec_gap = AudioSegment.silent(duration=500)

            for chunk in chunks[1:]:
                final_audio += one_sec_gap + chunk
        else:
            # Fallback in case a file has zero pauses longer than 1 second
            final_audio = normalized_audio

        new_duration = len(final_audio) / 1000.0
        time_saved = original_duration - new_duration

        # --- 5. EXPORT FOR TTS ---
        # Enforce the strict 22050 Hz Mono requirement during export
        final_audio.export(out_path, format="wav", parameters=["-ar", "22050", "-ac", "1"])

        # --- LOGGING ---
        if time_saved > 0.1:
            print(f"✂️ {filename}: Reduced from {original_duration:.1f}s to {new_duration:.1f}s (Trimmed {time_saved:.1f}s of dead air)")
        else:
            print(f"✅ {filename}: Processed (No long silences detected)")

    print(f"\n🎉 All preprocessed audio successfully saved to: {output_dir}")

In [5]:
# Point this to the folder where your 100 handpicked files live
preprocess_aphasia_audio("/rds/general/user/ak8224/home/emoji_project/data/Aphasia/final")

🎧 Found 100 files. Starting normalization and silence cropping...

✅ patient1_0496.wav: Processed (No long silences detected)
✅ patient1_0342.wav: Processed (No long silences detected)
✅ patient1_0331.wav: Processed (No long silences detected)
✅ patient1_0405.wav: Processed (No long silences detected)
✅ patient1_0082.wav: Processed (No long silences detected)
✅ patient1_0233.wav: Processed (No long silences detected)
✅ patient1_0432.wav: Processed (No long silences detected)
✅ patient1_0250.wav: Processed (No long silences detected)
✅ patient1_0312.wav: Processed (No long silences detected)
✅ patient1_0001.wav: Processed (No long silences detected)
✅ patient1_0245.wav: Processed (No long silences detected)
✅ patient1_0478.wav: Processed (No long silences detected)
✅ patient1_0390.wav: Processed (No long silences detected)
✅ patient1_0266.wav: Processed (No long silences detected)
✅ patient1_0391.wav: Processed (No long silences detected)
✅ patient1_0513.wav: Processed (No long silences

## Create Final Metadata (LJSpeech Format)

In [11]:
import os
import glob

# --- 1. SETUP PATHS ---
hpc_root = "/rds/general/user/ak8224/home"

# Directory containing your final 100 preprocessed audio files
wavs_dir = f"{hpc_root}/emoji_project/data/Aphasia/final/preprocessed"

# Directory containing your cleaned text scripts
scripts_dir = f"{hpc_root}/emoji_project/data/Aphasia/2"

# Where to save the final metadata file
output_metadata = f"{hpc_root}/emoji_project/data/Aphasia/final/preprocessed/metadata.csv"

# --- 2. GATHER AUDIO FILES ---
wav_files = glob.glob(os.path.join(wavs_dir, "*.wav"))
print(f"🔍 Found {len(wav_files)} audio files in the preprocessed directory.")

metadata_lines = []
missing_scripts = 0

# --- 3. MATCH AND BUILD ---
for wav_path in wav_files:
    # Extract just the filename without the .wav extension (e.g., "patient_0001")
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    
    # Build the path to where the matching text file *should* be
    txt_path = os.path.join(scripts_dir, f"{base_name}.txt")
    
    # Check if the text file actually exists in the script directory
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            # Read the transcript and strip any accidental whitespace/newlines
            transcript = f.read().strip()
            
        # Format: <absolute_wav_path>|<speaker_id>|<transcript>
        line = f"{wav_path}|0|{transcript}"
        metadata_lines.append(line)
    else:
        print(f"⚠️ Warning: Could not find matching transcript for {base_name}.wav")
        missing_scripts += 1

# --- 4. SAVE TO CSV ---
with open(output_metadata, "w", encoding="utf-8") as f:
    for line in metadata_lines:
        f.write(line + "\n")

# --- 5. REPORTING ---
print(f"\n✅ Successfully generated metadata.csv with {len(metadata_lines)} entries!")
if missing_scripts > 0:
    print(f"ℹ️ Skipped {missing_scripts} audio files because their text files were missing.")
print(f"📄 Saved to: {output_metadata}")

🔍 Found 100 audio files in the preprocessed directory.

✅ Successfully generated metadata.csv with 100 entries!
📄 Saved to: /rds/general/user/ak8224/home/emoji_project/data/Aphasia/final/preprocessed/metadata.csv


## Create Split for Fine-Tuning

In [3]:
import random

metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/metadata.csv"

with open(metadata_path, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

random.shuffle(lines)

split_idx = int(len(lines) * 0.95)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train.txt", "w") as f:
    f.write("\n".join(train_lines) + "\n")

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/val.txt", "w") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"✅ Created train.txt ({len(train_lines)} files) and val.txt ({len(val_lines)} files)!")

✅ Created train.txt (95 files) and val.txt (5 files)!
